# 06 - BERT Sentiment Classification (Objective 6, part 2)
Fine-tune DistilBERT (fast) for Negative / Neutral / Positive. To use full BERT or RoBERTa, change `MODEL_NAME`.

> Use a GPU (Google Colab: *Runtime > Change runtime type > T4 GPU*). On CPU it works, but is slow.

In [ ]:
%pip install -q transformers datasets accelerate torch scikit-learn pandas seaborn matplotlib pyarrow

In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding, set_seed)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
set_seed(SEED)
MODEL_NAME = "distilbert-base-uncased"   # or "bert-base-uncased", "roberta-base"
MAX_LEN, EPOCHS, BATCH, LR = 128, 3, 16, 3e-5

Path("data/outputs").mkdir(parents=True, exist_ok=True)
Path("models").mkdir(exist_ok=True)
print("GPU available:", torch.cuda.is_available())

In [ ]:
train = pd.read_parquet("data/processed/train.parquet")
val = pd.read_parquet("data/processed/val.parquet")
test = pd.read_parquet("data/processed/test.parquet")

label2id = {"negative": 0, "neutral": 1, "positive": 2}
id2label = {v: k for k, v in label2id.items()}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ReviewDataset(torch.utils.data.Dataset):
    def __init__(self, df):
        # text_light: BERT needs natural sentences, punctuation and negations
        self.enc = tokenizer(df["text_light"].tolist(), truncation=True, max_length=MAX_LEN)
        self.labels = df["label"].map(label2id).tolist()
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = self.labels[i]
        return item

train_ds, val_ds, test_ds = ReviewDataset(train), ReviewDataset(val), ReviewDataset(test)

## Handle class imbalance (Neutral is usually the smallest class) with a weighted loss

In [ ]:
weights = compute_class_weight("balanced", classes=np.array([0, 1, 2]), y=np.array(train_ds.labels))
class_weights = torch.tensor(weights, dtype=torch.float)
print("class weights:", dict(zip(label2id, weights.round(2))))

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights.to(outputs.logits.device))
        loss = loss_fn(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    return {"accuracy": accuracy_score(labels, preds), "precision_macro": p, "recall_macro": r, "f1_macro": f1}

## Train

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3, id2label=id2label, label2id=label2id)

args = TrainingArguments(
    output_dir="models/checkpoints",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=32,
    learning_rate=LR,
    weight_decay=0.01,
    eval_strategy="epoch",          # older transformers versions: evaluation_strategy="epoch"
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    seed=SEED,
    report_to="none",               # notebook 08 adds MLflow tracking
)

trainer = WeightedTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                          data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=compute_metrics)
trainer.train()

## Evaluate on the held-out test set (once)

In [ ]:
pred_output = trainer.predict(test_ds)
y_true = pred_output.label_ids
y_pred = pred_output.predictions.argmax(axis=-1)
names = [id2label[i] for i in range(3)]

print(classification_report(y_true, y_pred, target_names=names, digits=3))

cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=names, yticklabels=names)
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("BERT - test confusion matrix"); plt.show()

In [ ]:
# compare with the best baseline from notebook 05
bert_f1 = precision_recall_fscore_support(y_true, y_pred, average="macro")[2]
bert_acc = accuracy_score(y_true, y_pred)
baseline = pd.read_csv("data/outputs/baseline_test_metrics.csv")
comparison = pd.concat([baseline, pd.DataFrame([{"model": MODEL_NAME, "accuracy": bert_acc, "f1_macro": bert_f1}])])
comparison.round(3)

In [ ]:
# save the model + test predictions (used for error analysis and topic sentiment)
trainer.save_model("models/bert_sentiment")
tokenizer.save_pretrained("models/bert_sentiment")

out = test[["review_id", "review_text", "text_light", "rating", "label"]].copy()
out["pred"] = [id2label[i] for i in y_pred]
out.to_parquet("data/outputs/bert_test_predictions.parquet", index=False)